# Bab 7: Membangun Aplikasi Chat
## Mulai Cepat Microsoft Foundry Models API

Notebook ini diadaptasi dari [Repositori Contoh Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang mencakup notebook yang mengakses layanan [Azure OpenAI](aoai-assignment.ipynb).

> **Catatan:** GitHub Models akan dihentikan pada akhir Juli 2026. Notebook ini sekarang menargetkan [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), yang menawarkan katalog model percobaan gratis yang sama dan pengalaman Azure AI Inference SDK.


# Ikhtisar  
"Model bahasa besar adalah fungsi yang memetakan teks ke teks. Diberikan sebuah string teks input, model bahasa besar mencoba memprediksi teks yang akan muncul berikutnya"(1). Notebook "quickstart" ini akan memperkenalkan pengguna pada konsep LLM tingkat tinggi, persyaratan paket inti untuk memulai dengan AML, pengenalan ringan pada desain prompt, dan beberapa contoh singkat dari berbagai kasus penggunaan. 


## Daftar Isi  

[Ikhtisar](#overview)  
[Cara menggunakan Layanan OpenAI](#how-to-use-openai-service)  
[1. Membuat Layanan OpenAI Anda](#1.-creating-your-openai-service)  
[2. Instalasi](#2.-installation)    
[3. Kredensial](#3.-credentials)  

[Kasus Penggunaan](#use-cases)    
[1. Meringkas Teks](#1.-summarize-text)  
[2. Mengklasifikasikan Teks](#2.-classify-text)  
[3. Menghasilkan Nama Produk Baru](#3.-generate-new-product-names)  
[4. Melatih Ulang Klasifikator](#4.fine-tune-a-classifier)  

[Referensi](#references)


### Bangun prompt pertama Anda  
Latihan singkat ini akan memberikan pengenalan dasar untuk mengirimkan prompt ke model di Microsoft Foundry Models untuk tugas sederhana "ringkasan".


**Langkah-langkah**:  
1. Instal perpustakaan `azure-ai-inference` di lingkungan python Anda, jika Anda belum melakukannya.  
2. Muat perpustakaan pembantu standar dan atur kredensial Microsoft Foundry Models Anda.  
3. Pilih model untuk tugas Anda  
4. Buat prompt sederhana untuk model  
5. Kirimkan permintaan Anda ke API model!


### 1. Instal `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Impor perpustakaan bantuan dan buat kredensial


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Menemukan model yang tepat  
Model seperti GPT-4o dan GPT-4o mini dapat memahami dan menghasilkan bahasa alami, dan tersedia di katalog Microsoft Foundry Models bersama model dari Meta, Mistral, Cohere, dan Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Desain Prompt  

"Keajaiban model bahasa besar adalah bahwa dengan dilatih untuk meminimalkan kesalahan prediksi ini atas sejumlah besar teks, model akhirnya belajar konsep yang berguna untuk prediksi tersebut. Misalnya, mereka belajar konsep seperti"(1):  

* cara mengeja  
* cara kerja tata bahasa  
* cara membuat parafrase  
* cara menjawab pertanyaan  
* cara melakukan percakapan  
* cara menulis dalam banyak bahasa  
* cara membuat kode  
* dll.  

#### Cara mengendalikan model bahasa besar  
"Dari semua input ke model bahasa besar, yang paling berpengaruh adalah prompt teks(1).  

Model bahasa besar dapat diberi prompt untuk menghasilkan output dalam beberapa cara:  

Instruksi: Beritahu model apa yang Anda inginkan  
Penyelesaian: Memancing model untuk menyelesaikan awal dari apa yang Anda inginkan  
Demonstrasi: Tunjukkan kepada model apa yang Anda inginkan, dengan salah satu:  
Beberapa contoh dalam prompt  
Ratusan atau ribuan contoh dalam dataset pelatihan fine-tuning"  



#### Ada tiga panduan dasar dalam membuat prompt:  

**Tampilkan dan jelaskan**. Jelaskan dengan jelas apa yang Anda inginkan melalui instruksi, contoh, atau kombinasi keduanya. Jika Anda ingin model mengurutkan daftar item secara alfabet atau mengklasifikasikan paragraf berdasarkan sentimen, tunjukkan bahwa itulah yang Anda inginkan.  

**Sediakan data berkualitas**. Jika Anda mencoba membuat pengklasifikasi atau membuat model mengikuti pola, pastikan ada cukup contoh. Periksa kembali contoh Anda — model biasanya cukup pintar untuk melihat kesalahan ejaan dasar dan memberikan respons, tetapi juga bisa menganggap ini disengaja dan itu dapat mempengaruhi respons.  

**Periksa pengaturan Anda.** Pengaturan temperature dan top_p mengontrol seberapa deterministik model dalam menghasilkan respons. Jika Anda meminta respons yang hanya memiliki satu jawaban benar, Anda ingin mengatur nilai ini lebih rendah. Jika Anda mencari respons yang lebih beragam, Anda mungkin ingin mengaturnya lebih tinggi. Kesalahan nomor satu yang dilakukan orang pada pengaturan ini adalah mengira bahwa itu adalah kontrol "kecerdasan" atau "kreativitas".  


Sumber: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Kirim!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Merangkum Teks  
#### Tantangan  
Merangkum teks dengan menambahkan 'tl;dr:' di akhir sebuah paragraf teks. Perhatikan bagaimana model memahami cara melakukan sejumlah tugas tanpa instruksi tambahan. Anda dapat bereksperimen dengan prompt yang lebih deskriptif daripada tl;dr untuk memodifikasi perilaku model dan menyesuaikan ringkasan yang Anda terima(3).  

Pekerjaan terbaru telah menunjukkan peningkatan substansial pada banyak tugas dan tolok ukur NLP dengan melakukan pra-pelatihan pada korpus teks besar diikuti dengan penyempurnaan pada tugas tertentu. Walaupun biasanya arsitektur tidak tergantung tugas, metode ini masih memerlukan dataset penyempurnaan khusus tugas dengan ribuan atau puluhan ribu contoh. Sebaliknya, manusia umumnya dapat melakukan tugas bahasa baru hanya dari beberapa contoh atau instruksi sederhana - sesuatu yang saat ini masih sangat sulit dilakukan oleh sistem NLP. Di sini kami menunjukkan bahwa memperbesar model bahasa secara signifikan meningkatkan performa tanpa tergantung tugas dengan sedikit contoh, terkadang bahkan mencapai tingkat yang bersaing dengan pendekatan penyempurnaan mutakhir sebelumnya.  



Tl;dr


# Latihan untuk beberapa kasus penggunaan  
1. Meringkas Teks  
2. Mengklasifikasikan Teks  
3. Menghasilkan Nama Produk Baru


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasifikasikan Teks  
#### Tantangan  
Klasifikasikan item ke dalam kategori yang diberikan saat waktu inferensi. Dalam contoh berikut, kami menyediakan baik kategori maupun teks untuk diklasifikasikan dalam prompt(*playground_reference). 

Pertanyaan Pelanggan: Halo, salah satu tombol pada keyboard laptop saya baru-baru ini rusak dan saya membutuhkan penggantinya:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Hasilkan Nama Produk Baru
#### Tantangan
Buat nama produk dari kata-kata contoh. Di sini kami menyertakan dalam prompt informasi tentang produk yang akan kami buat namanya. Kami juga menyediakan contoh serupa untuk menunjukkan pola yang ingin kami terima. Kami juga telah mengatur nilai temperatur tinggi untuk meningkatkan keacakan dan respons yang lebih inovatif.

Deskripsi produk: Pembuat milkshake rumahan
Kata benih: cepat, sehat, kompak.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Deskripsi produk: Sepasang sepatu yang bisa muat untuk semua ukuran kaki.
Kata benih: dapat disesuaikan, pas, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referensi  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Praktik terbaik untuk penyetelan GPT models agar dapat mengklasifikasi teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lebih Lanjut  
[Tim Komersialisasi OpenAI](AzureOpenAITeam@microsoft.com) 


# Kontributor
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
